# TM1py Fundamentals

This module is the first in a sequence on tm1py, the Python library for
working with TM1 and Planning Analytics. Readers are assumed to be expert
in TM1, comfortable with cubes, dimensions, hierarchies, subsets, views,
MDX, rules, TurboIntegrator processes, and chores, and to have only
passing acquaintance with Python. The focus here is on what tm1py is, how
it connects to a TM1 server, and the shape of the read, transform, and
write pattern that makes the library worth learning in the first place.

tm1py wraps the TM1 REST API. Where a TI process runs on the server and
is constrained to the TI language and its data types, a tm1py script runs
in an ordinary Python process. It pulls data out of TM1, manipulates it
with the full Python ecosystem (pandas, NumPy, scikit-learn, your own
modules), and writes results back. The two are not in competition; they
solve different parts of a workflow. A common architecture is for tm1py
to handle the analytical transformation, then trigger a TI process for
the heavy server side write.

The topics below are arranged linearly for review. Structural grouping
(sections, chapters) can be applied later. A single running model, a
small `Sales Plan` cube with `Period`, `Region`, `Product`, `Version`,
and `Measure` dimensions, threads through every example, so each topic
introduces exactly one new idea.

---

## Topic list

1. tm1py and its place in the TM1 toolchain
2. Installing and importing
3. Connecting with TM1Service
4. Authentication modes
5. The service namespace
6. Reading the model: cubes, dimensions, elements
7. Reading cell data
8. Writing cell data
9. Running TI processes from Python
10. Errors and exceptions
11. The read, transform, write pattern
12. Real-world design principles
13. Common mistakes

---

## 1. tm1py and its place in the TM1 toolchain

A TM1 administrator already has several ways to move data through the
model. Architect and PAW provide interactive editing, TurboIntegrator
runs scripted loads on the server, rules and feeders compute derived
values, and the REST API exposes everything to outside callers. tm1py
sits on the REST layer, on the client side, and offers a Python facade
over it.

The library returns Python objects rather than HTTP responses, handles
authentication and pagination, and parses cell data into dictionaries
and pandas DataFrames. The user never writes a URL, never assembles a
JSON body, and rarely needs to think about the OData verbs underneath.
A call such as `tm1.cubes.get_all_names()` is a single line; the
equivalent against the raw REST API is several lines of HTTP plumbing.

Where tm1py beats TI: anything that benefits from the broader Python
ecosystem. Time series forecasting with statsmodels, optimization with
scipy, machine learning with scikit-learn, calls to external APIs, joins
against non TM1 data sources (SQL, parquet, Excel), or any manipulation
that pandas expresses more directly than TI's row by row data sources.
tm1py also wins when the transformation reads from many cubes, joins
their results, and writes back to a third, which is awkward in TI.

Where TI still wins: tight loops over many cells. TI executes on the
server, in process with the cube, and can write tens of thousands of
cells per second without touching a network. tm1py, however efficiently
batched, still pays for HTTP. A typical pattern is to do the analytical
work in tm1py, stage results in a small staging cube or a flat file, and
trigger a TI to fan the results out into the target cube.

Where rules still win: any value that can be expressed as a function of
other cells in the same cube. A rule recomputes automatically; a tm1py
write is a snapshot that goes stale the moment the inputs change.

## 2. Installing and importing

tm1py is a pure Python package on PyPI. Installation is one line:

In [ ]:
%%bash
pip install TM1py

Note the casing on the import. The package on PyPI is `TM1py`, the
import name is also `TM1py`, and the main class is `TM1Service`.

In [ ]:
from TM1py import TM1Service

There is no compiled component, no native dependency, no separate driver
to configure. tm1py talks to the TM1 server over HTTPS using the
`requests` library and JSON, and works against any TM1 server that
exposes the REST API (TM1 11.x and later, including all current
Planning Analytics releases on premises and on cloud).

The library is actively maintained by the community on GitHub. New
versions arrive every few months and track new TM1 features. Pinning a
specific version in `requirements.txt` (`TM1py==2.0.4`) is the right
practice for production scripts; allowing the latest is fine for ad hoc
analysis.

## 3. Connecting with TM1Service

`TM1Service` is the entry point. The constructor takes connection
parameters, opens a session against the TM1 server, and returns a live
service object whose attributes (covered in Topic 5) expose every
operation the library supports.

In [ ]:
from TM1py import TM1Service

tm1 = TM1Service(
    address="tm1.example.com",
    port=8010,
    user="admin",
    password="apple",
    ssl=True,
)

print(tm1.server.get_server_name())   # 'tm1srv01'
tm1.logout()

A live `TM1Service` holds an open session on the server, identified by a
session cookie. Sessions consume a slot in the server's session table
and have a configured timeout. Leaking sessions, leaving them open and
unused, eventually fills the slot pool and starts rejecting new
connections. Cleaning up after a script is therefore not optional.

The idiomatic way to handle cleanup in Python is the `with` block. A
`with` block guarantees that a designated cleanup runs when the block
exits, whether normally or by exception.

In [ ]:
with TM1Service(address="tm1.example.com", port=8010,
                user="admin", password="apple", ssl=True) as tm1:
    cubes = tm1.cubes.get_all_names()
    print(cubes)
# session is closed here, even if the body raised

The `with` form replaces the explicit `tm1.logout()` and is the only
form that survives unexpected exits cleanly. A script that calls
`logout` at the end of `try` will leak its session whenever the body
throws. The `with` block has no such hole.

For interactive notebook work, holding a long lived service object is
acceptable, but production code should always use `with`.

## 4. Authentication modes

The `TM1Service` constructor accepts the parameters needed for each TM1
authentication mode. Most installations use one of three.

**Basic authentication** is plain user and password against the TM1
internal user store. This is the most common setup for development and
small deployments.

In [ ]:
TM1Service(
    address="tm1.example.com", port=8010,
    user="admin", password="apple",
    ssl=True,
)

**CAM (IBM Cognos Analytics)** authenticates against a Cognos namespace.
The constructor takes a `namespace` alongside the usual user and
password.

In [ ]:
TM1Service(
    address="paw.example.com", port=8010,
    user="finance_user", password="...",
    namespace="LDAP",
    ssl=True,
)

When the caller already holds a preauthenticated Cognos passport, the
`cam_passport` argument takes that token instead of a password.

**Integrated Windows authentication (Mode 2)** uses the current
operating system credentials, with no password in code. Mode 5, by
contrast, is CAM combined with Integrated Login and is configured the
same way as Mode 4 plus `integrated_login=True`.

In [ ]:
TM1Service(
    address="tm1.example.com", port=8010,
    integrated_login=True,
    ssl=True,
)

Other modes (API keys, IBM Cloud OAuth, impersonation) are exposed as
additional constructor arguments. The TM1 administrator is the source of
truth for which mode applies; if there is no guidance, basic over SSL is
the default to try first.

`ssl=True` should be the default for any deployment beyond a local sandbox.
The only reason to set `ssl=False` is a development server with
HTTPS turned off, in which case `verify=False` may also be needed to
ignore self-signed certificates.

## 5. The service namespace

A `TM1Service` instance exposes the TM1 model as a tree of attributes,
one per object type or area of concern. Every attribute is itself a
service object with methods that map cleanly to REST verbs.

In [ ]:
tm1.cubes        # cube CRUD and metadata
tm1.dimensions   # dimensions
tm1.hierarchies  # hierarchies (most dimensions have one, named the same)
tm1.elements     # elements within a hierarchy
tm1.subsets      # named element subsets
tm1.views        # named MDX or native views
tm1.cells        # the actual cell data: read and write
tm1.processes    # TI processes
tm1.chores       # scheduled chains of processes
tm1.security     # users, groups, permissions
tm1.applications # the Apps tree visible in PAW
tm1.threads      # server threads (the equivalent of TM1 Top)
tm1.sandboxes    # personal write-back sandboxes
tm1.server       # server name, configuration, license info
tm1.git          # git integration (Planning Analytics 2.0.9.18+)

Methods on each service follow a verb convention that mirrors REST.

- `get_all_names()` returns a list of names.
- `get_all()` returns a list of objects.
- `get(name)` returns one object by name.
- `exists(name)` returns a bool.
- `create(obj)` creates one.
- `update(obj)` overwrites one.
- `update_or_create(obj)` upserts.
- `delete(name)` removes one.

Once the convention is internalized, most of tm1py is predictable. To
list every cube, `tm1.cubes.get_all_names()`. To check whether a process
exists, `tm1.processes.exists("Load_Sales")`. To delete a subset, call
`tm1.subsets.delete(subset_name="EU only", dimension_name="Region",
hierarchy_name="Region")`. There is no need to look up the exact method
name from documentation in the common cases.

## 6. Reading the model: cubes, dimensions, elements

The TM1 metadata, the structure of cubes, dimensions, hierarchies, and
elements, comes back as Python objects. Each object is a thin data
wrapper, comparable to a `dataclass` or a record: it has named
attributes and serializes cleanly to and from JSON, but does not contain
business logic.

In [ ]:
with TM1Service(**creds) as tm1:
    cube = tm1.cubes.get("Sales Plan")
    print(cube.dimensions)
    # ['Period', 'Region', 'Product', 'Version', 'Measure']

    dim = tm1.dimensions.get("Region")
    print(list(dim.hierarchies.keys()))
    # ['Region']                            # default hierarchy has the same name

    regions = tm1.elements.get_element_names("Region", "Region")
    print(regions[:3])
    # ['Europe', 'Americas', 'Asia']

Element lists are the most frequently fetched piece of metadata. They
are also stable within a session: regions added or renamed in TM1 do
not propagate to a Python set already in memory. The right pattern is to
fetch each list once at the start of a session and reuse it; refetch only
on a long running process where the model could change underneath.

In [ ]:
with TM1Service(**creds) as tm1:
    regions  = set(tm1.elements.get_element_names("Region",  "Region"))
    products = set(tm1.elements.get_element_names("Product", "Product"))
    periods  = set(tm1.elements.get_element_names("Period",  "Period"))
    # use these sets for the rest of the session

Beyond the flat element list, `tm1.elements.get_elements(...)` returns
full `Element` objects with type (Numeric, String, Consolidated),
attributes, and component children. `tm1.hierarchies.get(...)` returns
the full hierarchy with edges. These richer reads are needed when the
script has to navigate parent-child relationships or read element
attributes; for plain set membership and isin checks, names alone are
enough.

## 7. Reading cell data

The single most common tm1py operation is pulling a slice of a cube into
Python. There are three idiomatic shapes.

A **named view** that already exists on the server is the cleanest
input. The view encapsulates the slice; the Python code names it.

In [ ]:
with TM1Service(**creds) as tm1:
    df = tm1.cells.execute_view_dataframe(
        cube_name="Sales Plan",
        view_name="2026 Plan by Region",
        private=False,
    )

df.head()
#    Period   Region    Product   Version  Measure   Value
# 0  2026Q1   Europe    Standard  Plan     Revenue   120000.0
# 1  2026Q1   Europe    Standard  Plan     Units       1200.0
# ...

The DataFrame has one column per dimension and a single `Value` column.
Filtering, pivoting, and reshaping are then ordinary pandas operations.

An **MDX query** is the right tool when the slice is dynamic, parameterized
by the calling code, or simply not worth promoting to a saved view.

In [ ]:
mdx = """
SELECT
    NON EMPTY {[Period].[2026Q1], [Period].[2026Q2]} ON ROWS,
    NON EMPTY {[Region].[Europe], [Region].[Asia]}   ON COLUMNS
FROM [Sales Plan]
WHERE ([Measure].[Revenue], [Version].[Plan], [Product].[Standard])
"""
df = tm1.cells.execute_mdx_dataframe(mdx)

The **raw cellset** form, `execute_view` or `execute_mdx`, returns a
dictionary keyed by the element tuple. It is rarely the right choice for
analytical work, where a DataFrame is more useful, but is the natural
input for code that writes back cell by cell or that targets a specific
small set of cells.

In TI, the same data is read with `CellGetN` inside a loop over a
view's data source. The tm1py form returns the entire result set in one
network round trip and is far faster for any non-trivial slice. For
small slices and simple lookups, both approaches work; for analytical
slices over thousands or millions of cells, the batched read is
non negotiable.

## 8. Writing cell data

Writing follows the same shape as reading: prefer a single batched
operation over a loop. tm1py exposes three write methods.

`write_value` writes a single cell, identified by its element tuple. It
is the equivalent of TI's `CellPutN` and useful for one-off updates.

In [ ]:
tm1.cells.write_value(
    value=125_000.0,
    cube_name="Sales Plan",
    element_tuple=("2026Q1", "Europe", "Standard", "Plan", "Revenue"),
)

`write` accepts a dictionary of `{element_tuple: value}` and writes them
all in one request.

In [ ]:
updates = {
    ("2026Q1", "Europe",   "Standard", "Plan", "Revenue"): 125_000.0,
    ("2026Q1", "Americas", "Standard", "Plan", "Revenue"): 260_000.0,
}
tm1.cells.write(cube_name="Sales Plan", cellset_as_dict=updates)

`write_dataframe` is the equivalent for a tidy DataFrame, with one
column per dimension and a single `Value` column. It is the form that
pairs naturally with `execute_view_dataframe`: read, transform, write.

In [ ]:
tm1.cells.write_dataframe(
    cube_name="Sales Plan",
    data=updated_plan,            # one column per dimension, plus 'Value'
    use_blob=True,                # blob upload, faster for large writes
)

`use_blob=True` uses an internal blob upload path that is several
multiples faster than the regular write for tens of thousands of cells
or more. It is the right default for any non-trivial write.

Writes can be staged into a sandbox rather than going straight to the
base values. Pass `sandbox_name="my_scenario"` to any write method, and
the values land in that sandbox; the base cube is unaffected until the
sandbox is committed. This is the same mechanism users see in PAW under
"personal workspace" and is the safest way to test a write end to end.

## 9. Running TI processes from Python

Some operations belong on the server. Bulk loads from a flat file, fan
outs into a target cube, anything that needs to run inside the cube's
process boundary for speed: these are TI's natural territory. tm1py
triggers a TI from Python and reports its outcome.

In [ ]:
success, status, error_log = tm1.processes.execute_with_return(
    process_name="Load_Sales_From_Staging",
    pYear=2026,
    pVersion="Plan",
)
print(success, status)
# True 'CompletedSuccessfully'

Parameters declared on the TI process are passed as keyword arguments
with the `p` prefix matching the TI convention. The return value is a
tuple of three: a boolean success flag, a status string
(`CompletedSuccessfully`, `CompletedWithMessages`, `Aborted`,
`HasMinorErrors`, `QuitCalled`), and the name of the server side error
log file if one was produced.

`execute_with_return` is synchronous: the call blocks until TI finishes.
For long running processes, set a generous request timeout on the
`TM1Service` constructor. For workflows that should not block the
caller, use `tm1.chores.execute(name)`, which kicks off a chore (a
chained TI sequence) and returns immediately.

The hybrid pattern is often the right architecture: tm1py orchestrates,
pulls data, runs the analytical transformation, writes results to a
small staging cube or a flat file, and then triggers a TI to fan those
results into the production cube. Each tool does what it is good at.

## 10. Errors and exceptions

Python signals failure by raising an exception. Code that needs to
recover wraps the risky call in a `try` / `except`; code that does not
recover lets the exception propagate, which prints a traceback and ends
the script. This differs from TI, where errors set a status code that
the process must check explicitly. In Python, the default is loud
failure.

tm1py raises `TM1pyException` (and its subclasses) for any error
returned by the TM1 server.

In [ ]:
from TM1py.Exceptions import TM1pyException

with TM1Service(**creds) as tm1:
    try:
        tm1.cells.write_value(
            value=100,
            cube_name="Sales Plan",
            element_tuple=("2026Q1", "Atlantis", "Standard", "Plan", "Revenue"),
        )
    except TM1pyException as exc:
        print(f"write failed: {exc}")
        # write failed: Status code 400 - Element 'Atlantis' not found in dimension 'Region'

The most common causes are familiar to TM1 administrators: a
misspelled element name (element names are case sensitive), a session
that has timed out, a locked cube, a malformed MDX query, or a
permission denied on the target cube. The error message from the
server is preserved in the exception and is usually enough to diagnose
the problem.

For interactive work, letting the exception propagate is fine; the
traceback ends in the offending call and the message comes with it. For
batch or production scripts, wrap the session boundary in a single
`try` / `except` that logs the failure and exits with a non-zero status.
Per-operation `try` / `except` only adds noise when there is no
meaningful recovery available.

## 11. The read, transform, write pattern

The single most useful tm1py workflow combines every previous topic. A
full end-to-end script reads a slice from a cube, transforms it with
pandas, writes the result back, and optionally triggers a TI for
downstream effects.

In [ ]:
import pandas as pd
from TM1py import TM1Service

with TM1Service(**creds) as tm1:
    # 1. Read: pull last year's actuals from Sales Plan
    actuals = tm1.cells.execute_view_dataframe(
        cube_name="Sales Plan",
        view_name="2025 Actuals",
        private=False,
    )

    # 2. Transform: project 2026 plan as 2025 actuals + 5% growth
    plan = (
        actuals
        .query("Measure == 'Revenue'")
        .assign(
            Period=lambda d: d["Period"].str.replace("2025", "2026"),
            Version="Plan",
            Value=lambda d: (d["Value"] * 1.05).round(2),
        )
    )

    # 3. Write: send the plan back into Sales Plan
    tm1.cells.write_dataframe(
        cube_name="Sales Plan",
        data=plan,
        use_blob=True,
    )

    # 4. Optional: kick off a TI to fan the plan into derived cubes
    success, status, _ = tm1.processes.execute_with_return(
        process_name="Recalculate_Plan_Aggregates",
    )
    if not success:
        raise RuntimeError(f"aggregation failed: {status}")

This is the killer use case for tm1py and the reason most TM1 teams
adopt it. The transformation step, three lines of pandas, would be a
fifty line TI process or a manual Excel manipulation. By moving the
analytical logic to Python while keeping the cube as the system of
record, the workflow gets the best of both tools.

The script reads top to bottom as a pipeline. The `with` block bounds
the session. The four steps are clearly labeled and could be replaced
independently. Production versions of this pattern add validation
(see the pandera module) before the write and structured logging around
each step, but the core skeleton stays the same.

## 12. Real-world design principles

**Always use `with TM1Service(...) as tm1`.** Manual `tm1.logout()` is
correct only if every code path is guaranteed to reach it. Exceptions,
early returns, and forgotten branches all leak sessions; the `with` block
does not. Reserving a special exception for "the session would not
close" is not worth the complexity when Python already provides a
deterministic mechanism.

**Read in batches, write in batches.** Single-cell reads and writes are
the slow path. A view or MDX query returns its entire result set in one
HTTP round trip; `write_dataframe` posts the entire payload in one
request. A loop that reads or writes one cell at a time pays the network
round trip cost on every iteration and is one or two orders of magnitude
slower for any non-trivial slice.

**Prefer named views over inline MDX where possible.** A named view is
a contract: the TM1 administrator sees it, the Python script consumes
it, and a change to the slice is a change in one place. Inline MDX
embedded in Python is fine for parameterized or one-off slices, but for
recurring loads, the view is the right home for the slice definition.

**Cache element lists per session.** Every `get_element_names` call is
a network round trip. Fetch each dimension's elements once at the start
of the session and reuse the resulting list or set. The same applies to
the cube's dimension list and to subsets that drive view definitions.

**Push heavy server side work to TI; keep analytical work in Python.**
Loops over millions of cells belong in TI, where they run in process
with the cube and avoid the network entirely. Joins, reshapes, model
fits, and external API calls belong in Python, where the ecosystem is
broader. Most non-trivial workflows use both tools, with tm1py
orchestrating.

**Validate at the boundary, not throughout.** A single
`pandera`-style validation right after the read catches dimension
mismatches, dtype drift, and bad element names before the
transformation runs. Sprinkling `assert` statements through the body
adds noise without adding safety; the boundary check is the place where
the data is suspect, and the body is the place where it should be
trusted.

**Treat element names as exact strings.** TM1 element names are case
sensitive in MDX, in REST, and therefore in tm1py. `"Europe"` and
`"europe"` are different elements. Round trip through the TM1 API
(or fetch from `get_element_names`) rather than typing names by hand
into Python literals; the round trip preserves the canonical casing.

## 13. Common mistakes

A short collection of errors that are easy to make and worth recognizing
early.

**Forgetting the `with` block, leaking sessions.** Every uncleaned
session sits in the server's session table until it times out. Over
days of development, the count climbs and starts rejecting connections.

In [ ]:
# Wrong: leaks on any exception in the body
tm1 = TM1Service(**creds)
df = tm1.cells.execute_view_dataframe(...)
tm1.logout()

# Correct
with TM1Service(**creds) as tm1:
    df = tm1.cells.execute_view_dataframe(...)

**Loading a whole cube without slicing.** `execute_mdx_dataframe` over
an unrestricted query against a multidimensional cube can return tens
of millions of rows and exhaust memory. Filter, slice with a view, or
use a MDX `WHERE` clause.

In [ ]:
# Wrong: loads the entire cube
mdx = "SELECT NON EMPTY [Sales Plan].Members ON ROWS FROM [Sales Plan]"
df = tm1.cells.execute_mdx_dataframe(mdx)

# Correct: scope to the period and version of interest
mdx = """
SELECT NON EMPTY {[Period].[2026Q1]} * {[Region].Members} ON ROWS,
       NON EMPTY {[Measure].[Revenue]} ON COLUMNS
FROM [Sales Plan] WHERE ([Version].[Plan])
"""
df = tm1.cells.execute_mdx_dataframe(mdx)

**Single cell writes in a loop.** A Python loop calling `write_value`
makes one HTTP round trip per cell. Batch with `write_dataframe` or
`write` instead.

In [ ]:
# Wrong: one round trip per row
for _, row in plan.iterrows():
    tm1.cells.write_value(
        value=row["Value"], cube_name="Sales Plan",
        element_tuple=tuple(row[dim_cols]),
    )

# Correct: one round trip total
tm1.cells.write_dataframe(cube_name="Sales Plan", data=plan, use_blob=True)

**Wrong element name capitalization.** Element names are case sensitive.
A literal `"europe"` in Python code does not match an element named
`"Europe"` in TM1, even though TM1 displays both consistently.

In [ ]:
# Wrong: silent miss in MDX, raises on write
tm1.cells.write_value(100, "Sales Plan", ("2026Q1", "europe", ...))

# Correct: pull canonical names from TM1 once
regions = set(tm1.elements.get_element_names("Region", "Region"))
assert "Europe" in regions

**Treating `execute_with_return` as fire and forget.** `execute_with_return`
blocks until the TI finishes. For long-running processes, set a generous
request timeout on the service. For genuinely asynchronous work, use a
chore.

In [ ]:
# Wrong assumption: this returns immediately
tm1.processes.execute_with_return(process_name="Long_Running_Load")
# the call blocks for the full duration of the TI

# Correct: a chore runs server side and returns at once
tm1.chores.execute(chore_name="Nightly_Load")

**Rebuilding `TM1Service` per call.** Every `TM1Service(...)` opens a
new session, with the cost of authentication and an HTTP handshake.
Inside a single script, build it once and reuse it.

In [ ]:
# Wrong: a new login on every call
def read_period(period):
    with TM1Service(**creds) as tm1:
        return tm1.cells.execute_mdx_dataframe(
            f"SELECT ... WHERE ([Period].[{period}])"
        )

dfs = [read_period(p) for p in periods]   # one login per period

# Correct: one login covers the whole batch
with TM1Service(**creds) as tm1:
    dfs = [
        tm1.cells.execute_mdx_dataframe(
            f"SELECT ... WHERE ([Period].[{p}])"
        )
        for p in periods
    ]

**Catching and swallowing `TM1pyException`.** A bare `except` that
discards the exception removes the only signal the script has that
something went wrong. Either let it propagate, or log the message before
deciding how to proceed.

In [ ]:
# Wrong: failure is invisible
try:
    tm1.cells.write_dataframe(cube_name="Sales Plan", data=plan)
except TM1pyException:
    pass

# Correct: log the message; reraise if not recoverable
try:
    tm1.cells.write_dataframe(cube_name="Sales Plan", data=plan)
except TM1pyException as exc:
    logger.error("plan write failed: %s", exc)
    raise